In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Mula pantas Executor

<Accordion>
<AccordionItem title="Versi pakej">

Kod di halaman ini dibangunkan menggunakan keperluan berikut.
Kami syorkan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

Seperti primitif [Sampler](/guides/get-started-with-sampler), Executor menyampel daftar output daripada pelaksanaan Circuit kuantum, tetapi ia tidak mempunyai sebarang penindasan atau pengurangan ralat terbina dalam. Sebaliknya, ia adalah sebahagian daripada [model pelaksanaan berarah](/guides/directed-execution-model) yang menyediakan bahan-bahan untuk menangkap niat reka bentuk di sisi klien, dan mengalihkan penjanaan variasi Circuit yang mahal ke sisi pelayan. Executor mengikuti arahan yang diberikan dalam anotasi Circuit dan pilihan, menjana dan mengikat nilai parameter, melaksanakan Circuit yang terikat pada perkakasan, dan mengembalikan keputusan pelaksanaan dan metadata. Ia tidak membuat sebarang keputusan implisit untuk anda dan memberikan anda kawalan dan ketelusan penuh.

> **Note:** Pakej Qiskit belum mempunyai kelas asas untuk primitif Executor.

## Sebelum anda mulakan
Sesetengah contoh kod di halaman ini menggunakan `samplex`, yang merupakan sebahagian daripada pakej Samplomatic. Oleh itu, sebelum menjalankan blok kod tersebut, anda mesti memasang Samplomatic, seperti yang ditunjukkan dalam blok kod berikut. Untuk maklumat lanjut, lihat [dokumentasi Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [2]:
print(backend)

<IBMBackend('ibm_fez')>


### 2. Create and transpile a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have parameters.

In [3]:
# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary
# classical registers.
circuit.measure_all()

The circuit needs to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [4]:
# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 2. Cipta dan transpail Circuit
Kamu memerlukan sekurang-kurangnya satu Circuit untuk menggunakan primitif Executor. Ia boleh mempunyai parameter secara pilihan.

In [5]:
# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

Circuit perlu ditransformasi untuk hanya menggunakan arahan yang disokong oleh QPU (dirujuk sebagai Circuit *set seni bina arahan (ISA)*). Gunakan Transpiler untuk melakukan ini.

In [6]:
# Generate a boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
boxes_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxes_pm.run(isa_circuit)
boxed_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/get-started-with-executor/extracted-outputs/245a4574-3ce9-4f77-98c8-af32cde8ac01-0.svg" alt="Output of the previous code cell" />

### 3. Mulakan `QuantumProgram`
Mulakan `QuantumProgram` dengan beban kerja anda. `QuantumProgram` terdiri daripada `QuantumProgramItems`. Biasanya, setiap item terdiri daripada Circuit, set nilai parameter, dan mungkin `samplex` untuk merawakkan kandungan Circuit. Untuk butiran penuh, lihat [Input dan output Executor](/guides/executor-input-output).

Sel berikut memulakan `QuantumProgram` dan menentukan untuk melakukan 25 tembakan. Seterusnya, ia menambahkan Circuit sasaran yang telah ditransail.

In [7]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

# Append the template circuit and samplex as a `samplex_item`
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    shape=(num_randomizations := 20,),
)

### 4. Pilihan: Kumpulkan Gate dan ukuran ke dalam kotak beranotasi
Mengumpulkan arahan ke dalam kotak dan membuat anotasi adalah cara utama untuk menentukan niat anda. Dalam contoh berikut, kita menggunakan `generate_boxing_pass_manager` dan parameter twirling-nya untuk mengumpulkan Gate dua-Qubit dan ukuran ke dalam kotak dan menerapkan anotasi twirling.

In [8]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d8286580bvlc73d1vmsg', 'executor')>

In [9]:
# Retrieve the result
result = job.result()

### 6. Panggil Executor dan dapatkan keputusan
Jalankan `QuantumProgram` pada Backend IBM&reg; menggunakan primitif `Executor` dengan pilihan lalai. Lihat [Pilihan Executor](/guides/executor-options) untuk mengetahui tentang pilihan yang tersedia.